# Cell 1: Environment Setup

In [4]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v6 FREQUENCY-BUDGET SWEEP — every choice below is FIXED (v5 settings) except
# the strongest-N budget N, which is the only variable. SEED is the single
# source for all seeding in this notebook.
# ==============================================================================
SEED = 42
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest-N (top-N mean |rFFT| bins, TRAINING audio only, frozen for val/test, per N)",
    "SWEPT VARIABLE N":    "N ∈ {32, 48, 64, 96, 128} (ascending; model audio_in_dim = N)",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; no Hann)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seed":                f"{SEED} (random / numpy / torch / cuda, cuDNN deterministic; reset for every N)",
    "Everything else":     "v5 unchanged (architecture except audio input dim, sensor path, split, 20 epochs, AdamW, eval)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v6 — FREQUENCY-BUDGET SWEEP CONFIGURATION")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction for every condition of the frequency-budget sweep (Cell 6).
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

  AEROSYNC-MAMBA v6 — FREQUENCY-BUDGET SWEEP CONFIGURATION
  • Audio bin selection : strongest-N (top-N mean |rFFT| bins, TRAINING audio only, frozen for val/test, per N)
  • SWEPT VARIABLE N    : N ∈ {32, 48, 64, 96, 128} (ascending; model audio_in_dim = N)
  • Audio windowing     : none (raw 1536-sample chunk -> rfft; no Hann)
  • Audio compression   : sqrt(magnitude)
  • Audio normalization : none
  • Classification loss : class-weighted cross-entropy (inverse TRAIN-set class frequency)
  • Seed                : 42 (random / numpy / torch / cuda, cuDNN deterministic; reset for every N)
  • Everything else     : v5 unchanged (architecture except audio input dim, sensor path, split, 20 epochs, AdamW, eval)
✅ Device: cuda (Tesla T4)
mamba_ssm already installed.
✅ Seeds set -> random=42, numpy=42, torch=42, cuda=42 (cuDNN deterministic)


# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba)

In [5]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2, d_state: int = 16):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        num_mamba_blocks: int = 4,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=4, beta=temporal_bias_scale)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        if _MAMBA_CLASS is not None:
            self.mamba_blocks = nn.ModuleList([
                RealMambaBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        else:
            self.mamba_blocks = nn.ModuleList([
                VectorizedSSMBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print("Bi-Modal AeroSync-Mamba model class defined successfully!")


FUSION BACKBONE RESOLUTION: mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected

Bi-Modal AeroSync-Mamba model class defined successfully!


# Cell 3: Bi-Modal Loss Function (Classification + Audio-IMU InfoNCE + KL)

In [6]:
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

Loss & Metrics helper ready!


# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset


In [7]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

N_BUDGETS = [32, 48, 64, 96, 128]  # strongest-N audio feature budgets swept in Cell 6 (ascending)
_MISSION_CACHE = {}  # mission_dir -> parse_real_mission(...) output, parsed ONCE and reused for every N

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


def get_mission(mission_dir: Path):
    """parse_real_mission, memoized: every mission's JSONL is parsed once and the
    same arrays are reused for every N (identical to re-parsing each time)."""
    if mission_dir not in _MISSION_CACHE:
        _MISSION_CACHE[mission_dir] = parse_real_mission(mission_dir)
    return _MISSION_CACHE[mission_dir]


# 2.3 Strongest-N FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once per N, before any train/val/test window slicing for
    that N, so the exact same frozen indices are reused identically across all
    three splits.

    Identical criterion to the v5 strongest-64 selection; only the budget N is
    a parameter now. Returns (sorted bin indices, data-derived bin width in Hz
    or None).
    """
    print(f"[STRONGEST-{n_bins}] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc=f"Bin selection N={n_bins} (train-only audio scan)", leave=False):
        mission_data = get_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        f"STRONGEST-{n_bins} FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-{n_bins} FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )
    bin_width_hz = est_sample_rate_hz / samples_per_buffer if est_sample_rate_hz else None

    print(f"[STRONGEST-{n_bins}] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-{n_bins}] Selected bin indices (ascending): {top_bins.tolist()}")
    if bin_width_hz:
        print(f"[STRONGEST-{n_bins}] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        # contiguous runs of selected bins -> approx. frequency bands
        runs, start = [], top_bins[0]
        for prev, cur in zip(top_bins[:-1], top_bins[1:]):
            if cur != prev + 1:
                runs.append((start, prev))
                start = cur
        runs.append((start, top_bins[-1]))
        print(f"[STRONGEST-{n_bins}] Selected bins as {len(runs)} contiguous band(s) -> approx. frequency (Hz):")
        for lo, hi in runs:
            print(f"    bins {lo:3d}-{hi:3d} ({hi - lo + 1:3d} bins)  ->  {lo * bin_width_hz:8.1f} - {hi * bin_width_hz:8.1f} Hz")
    else:
        print(f"[STRONGEST-{n_bins}] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins, bin_width_hz


# 2.4 Audio feature extraction — the single, permanent path (v5), width = N:
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-N bins -> sqrt
#     (no normalization).
def compute_audio_features(audio_chunks, bin_indices):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    bin_indices  : frozen strongest-N bin indices from select_strongest_audio_bins
                   (computed once per N from TRAINING audio; never recomputed here)
    returns      : (n_chunks, N) float32 = sqrt(|rfft|) at those bins, N = len(bin_indices)
    """
    assert bin_indices is not None and len(bin_indices) > 0, (
        "FEATURES FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )
    n_bins = len(bin_indices)

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = bin_indices[bin_indices < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, n_bins), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, N={n_bins})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, bin_indices, window_sec=2.56, step_sec=1.28, Ta=64, Ts=128):
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert bin_indices is not None and len(bin_indices) > 0, (
        "SLICE FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest-N audio features: FFT magnitude spectrum, frozen top-N bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-N bins -> sqrt, via compute_audio_features (2.4).
    n_bins = len(bin_indices)
    a_feats = compute_audio_features(audio, bin_indices)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"STRONGEST-{n_bins} FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {n_bins})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-{n_bins} VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-{n_bins} bins) shape: {a_feats.shape}")
        print(f"[STRONGEST-{n_bins} VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              f"(frequency-domain magnitude spectrum) using the frozen strongest-{n_bins} bin indices, "
              f"NOT the first {n_bins} bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, n_bins), dtype=np.float32)
        for c in range(n_bins):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, n_bins), (
            f"STRONGEST-{n_bins} FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {n_bins})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats", "N_AUDIO_BINS", "SELECTED_BIN_INDICES") if n in globals()]

assert _feat_params == ["audio_chunks", "bin_indices"], f"compute_audio_features has unexpected parameters: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v6 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no hardcoded 64-bin globals defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- 5. Per-budget preprocessing (called once per N by run_experiment in Cell 6) ---
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
_heldout_missions = {m for mp in (val_mission_map, test_mission_map) for ms in mp.values() for m in ms}
assert not (set(all_train_missions) & _heldout_missions), (
    "LEAKAGE: training missions overlap validation/test missions")

def preprocess_for_budget(n_bins):
    """Strongest-N bins from TRAINING missions only (frozen) -> train/val/test
    windows built with those same frozen indices -> cached .pt files for N."""
    global _fft_shape_printed
    _fft_shape_printed = False

    # --- Freeze strongest-N bin indices from TRAINING missions only, BEFORE any
    # train/val/test window preprocessing happens. ---
    bin_indices, bin_width_hz = select_strongest_audio_bins(all_train_missions, n_bins=n_bins)
    _bin_fingerprint_at_selection = (bin_indices[:5].tolist(), bin_indices[-5:].tolist())

    split_maps = [("TRAIN", train_mission_map), ("VAL", val_mission_map), ("TEST", test_mission_map)]
    split_samples = {}
    _fp_train = None
    for split_label, mission_map in split_maps:
        note = "(frozen strongest-N bins)" if split_label == "TRAIN" else "(same frozen bin indices — NOT recomputed)"
        print(f"\n--- Preprocessing {split_label} split, N={n_bins} {note} ---")
        samples, n_missions = [], 0
        for c_label in sorted(class_missions.keys()):
            for m in tqdm(mission_map[c_label], desc=f"{split_label.capitalize()} Class {c_label} (N={n_bins})", leave=False):
                p = get_mission(m)
                if p:
                    samples.extend(slice_real_bimodal(p, c_label, bin_indices))
                    n_missions += 1
        split_samples[split_label] = samples

        _fp = (bin_indices[:5].tolist(), bin_indices[-5:].tolist())
        if split_label == "TRAIN":
            _fp_train = _fp
            print(f"[STRONGEST-{n_bins}] Frozen bin indices applied @ TRAIN (first5, last5): {_fp} "
                  f"| matches selection: {_fp == _bin_fingerprint_at_selection}")
        else:
            print(f"[STRONGEST-{n_bins}] Frozen bin indices applied @ {split_label:<5} (first5, last5): {_fp} "
                  f"| matches TRAIN: {_fp == _fp_train}")
        # compute_audio_features asserted (n_chunks, N) for every one of these missions
        print(f"[SHAPE CHECK] {split_label}: audio feature array asserted == (n_chunks, {n_bins}) "
              f"for all {n_missions} missions ✅")

    # --- Final shape verification across all three splits ---
    for split_label, samples in split_samples.items():
        if samples:
            x_a_shapes = {tuple(s["x_a"].shape) for s in samples}
            assert x_a_shapes == {(64, n_bins)}, (
                f"STRONGEST-{n_bins} FAILED: {split_label} x_a shapes inconsistent: {x_a_shapes}, "
                f"expected only (64, {n_bins})"
            )
    print(f"[STRONGEST-{n_bins}] Verified: every train/val/test window's audio feature array "
          f"has shape (Ta=64, N={n_bins}).")

    print("\n" + "=" * 60)
    print(f"     100% REAL PREPROCESSING COMPLETE (N={n_bins}, ZERO SYNTHETIC DATA)   ")
    if _chunk_duration_log:
        _arr = np.array(_chunk_duration_log)
        print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
    print("=" * 60)
    print(f"  • Total Train Diagnostic Windows : {len(split_samples['TRAIN'])}")
    print(f"  • Total Val Diagnostic Windows   : {len(split_samples['VAL'])}")
    print(f"  • Total Test Diagnostic Windows  : {len(split_samples['TEST'])}")
    print("=" * 60)

    out_dir = CACHE_DIR / f"N{n_bins}"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": out_dir / "bimodal_train.pt", "val": out_dir / "bimodal_val.pt", "test": out_dir / "bimodal_test.pt"}
    torch.save(split_samples["TRAIN"], paths["train"])
    torch.save(split_samples["VAL"], paths["val"])
    torch.save(split_samples["TEST"], paths["test"])
    print(f"✅ Cached real bimodal dataset for N={n_bins} in {out_dir}")

    return {
        "bin_indices": bin_indices,
        "bin_width_hz": bin_width_hz,
        "freq_min_hz": float(bin_indices.min() * bin_width_hz) if bin_width_hz else None,
        "freq_max_hz": float(bin_indices.max() * bin_width_hz) if bin_width_hz else None,
        "n_train_missions": len(all_train_missions),
        "n_windows": (len(split_samples["TRAIN"]), len(split_samples["VAL"]), len(split_samples["TEST"])),
        "paths": paths,
    }

print(f"\n✅ Preprocessing helpers ready — mission split fixed above; strongest-N selection + window "
      f"preprocessing run per condition in Cell 6 for N = {N_BUDGETS}.")

📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...
Cloning into '/kaggle/working/tii_uav_dataset'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 324 (delta 111), reused 321 (delta 111), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 1.15 GiB | 27.90 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Updating files: 100% (217/217), done.
✅ Dataset cloned successfully!
[NO-MODE-SWITCH CHECK] compute_audio_features('audio_chunks', 'bin_indices') | slice_real_bimodal has no window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif branches: clean ✅
[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no hardcoded 64-bin globals defined ✅
Class 0 (19 flights) -> Train: 13, Val: 2, Test: 4
Class 1 (20 flights) -> Train: 14, Val: 3, Test: 3
Class 2 (20 flights) -> Train: 14, Val: 3,

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [8]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = 64
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

batch_size = 32

def build_dataloaders(paths, batch_size=batch_size):
    """Datasets + loaders for one N (same settings as v5)."""
    train_ds = RealBiModalDataset(paths["train"])
    val_ds   = RealBiModalDataset(paths["val"], mean=train_ds.mean, std=train_ds.std)
    test_ds  = RealBiModalDataset(paths["test"], mean=train_ds.mean, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

print("RealBiModalDataset / build_dataloaders ready (DataLoaders are built per N in Cell 6).")

RealBiModalDataset / build_dataloaders ready (DataLoaders are built per N in Cell 6).


In [9]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
def verify_real_taus(train_ds):
    print("=" * 70)
    print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
    print("=" * 70)

    check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
    taus_a, taus_s = [], []
    for idx in check_idxs:
        item = train_ds[idx]
        taus_a.append(item["tau_a"])
        taus_s.append(item["tau_s"])
        print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

    all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
    all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

    assert not all_tau_a_identical, (
        "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    assert not all_tau_s_identical, (
        "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
    print("=" * 70)

print("verify_real_taus(train_ds) ready — runs for every N inside run_experiment.")

verify_real_taus(train_ds) ready — runs for every N inside run_experiment.


# Cell 6: run_experiment(N) — one isolated condition of the frequency-budget sweep

In [10]:
# ==============================================================================
# run_experiment(N) — ONE complete, isolated condition of the frequency-budget
# sweep:
#   seeds reset -> strongest-N bins from TRAINING audio only (frozen) ->
#   train/val/test windows -> DataLoaders -> FRESH model with audio_in_dim=N +
#   class-weighted loss + optimizer + scheduler -> 20-epoch training with
#   best-val checkpoint -> unseen-test evaluation -> GPU memory freed.
# Everything except N is the v5 pipeline unchanged (same hyperparameters, same
# training loop, same RNG order: seed reset immediately before model
# construction, exactly as v5 Cell 6 did).
# ==============================================================================
import gc

EPOCHS = 20

def run_experiment(n_bins, seed=SEED, epochs=EPOCHS):
    t_run0 = time.time()
    set_all_seeds(seed)
    print(f"[SEEDS] reset at condition start -> random / numpy / torch / cuda = {seed}")

    # --- 1. Strongest-N selection (TRAIN audio only) + window preprocessing ---
    prep = preprocess_for_budget(n_bins)
    if prep["bin_width_hz"]:
        freq_str = (f"{prep['freq_min_hz']:.1f} Hz (bin {prep['bin_indices'].min()}) – "
                    f"{prep['freq_max_hz']:.1f} Hz (bin {prep['bin_indices'].max()})")
    else:
        freq_str = f"bins {prep['bin_indices'].min()}–{prep['bin_indices'].max()} (Hz estimate unavailable)"
    print("\n" + "-" * 90)
    print(f"[CONDITION] N = {n_bins} strongest bins")
    print(f"[CONDITION] frequency range covered by the {n_bins} selected bins (min – max): {freq_str}")
    print(f"[CONDITION] bins selected from TRAINING-SET audio only ({prep['n_train_missions']} train "
          f"missions; 0 of {len(_heldout_missions)} validation/test missions used) and frozen for val/test ✅")
    print("-" * 90 + "\n")

    # --- 2. Datasets / loaders (v5 settings) ---
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_dataloaders(prep["paths"])
    for split_name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
        xa_shape = tuple(ds[0]["x_a"].shape)
        assert xa_shape == (64, n_bins), f"[N={n_bins}] {split_name} x_a is {xa_shape}, expected (64, {n_bins})"
    verify_real_taus(train_ds)

    # --- 3. FRESH model sized for this N (seed reset right before, as in v5 Cell 6) ---
    set_all_seeds(seed)
    model = BiModalAeroSyncMamba(
        d_model=256,
        n_classes=5,
        num_mamba_blocks=4,
        dropout=0.15,
        temporal_bias_scale=1.0,
        audio_in_dim=n_bins,
        sensor_in_dim=6,
    ).to(device)
    audio_in = model.audio_encoder[0].in_features
    assert audio_in == n_bins, f"model audio input is {audio_in}, expected N={n_bins}"
    print(f"[MODEL] fresh model | audio_encoder input = {audio_in} (= N ✅) | "
          f"params = {sum(p.numel() for p in model.parameters()):,}")

    _fusion_cls = type(model.mamba_blocks[0])
    print(f"[MODEL] FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}"
          + ("  🚨 pure-PyTorch FALLBACK, NOT real mamba_ssm 🚨" if _fusion_cls.__name__ == "VectorizedSSMBlock"
             else "  ✅ REAL mamba_ssm selective-scan SSM"))

    # FIX 4: inverse-frequency class weights from REAL training-set class counts
    train_labels_arr = np.array([s["label"] for s in train_ds.samples])
    class_counts = np.bincount(train_labels_arr, minlength=5)
    class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
    print("[FIX 4] class weights (from real train counts): "
          + " | ".join(f"C{c}: n={class_counts[c]} w={class_weights_np[c]:.4f}" for c in range(5)))

    criterion = BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1, class_weights=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_f1 = 0.0
    best_weights = None
    best_epoch = None
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": []}
    ckpt_path = CACHE_DIR / f"aerosync_bimodal_best_N{n_bins}.pt"

    try:
        # --- 4. Training (identical loop to v5 Cell 6) ---
        start_time = time.time()
        print(f"🚀 Training Bi-Modal AeroSync-Mamba (N={n_bins}) on {len(train_ds)} real flight windows...\n")

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, running_cls = 0.0, 0.0

            pbar = tqdm(train_loader, desc=f"N={n_bins} Epoch {epoch:02d}/{epochs}", leave=False)
            for batch in pbar:
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer.zero_grad()
                logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                running_cls += l_dict["loss_cls"]
                pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

            scheduler.step()
            avg_train_loss = running_loss / len(train_loader)

            # --- Validation ---
            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            val_loss_total = 0.0

            with torch.no_grad():
                for batch in val_loader:
                    xa = batch["x_a"].to(device)
                    xs = batch["x_s"].to(device)
                    ta = batch["tau_a"].to(device)
                    ts = batch["tau_s"].to(device)
                    labels = batch["label"].to(device)

                    logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                    v_loss, _ = criterion(logits, labels, pooled_reps, aux_preds)
                    val_loss_total += v_loss.item()

                    probs = F.softmax(logits, dim=-1)
                    val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    val_targets.extend(labels.cpu().numpy())
                    val_probs.extend(probs.cpu().numpy())

            avg_val_loss = val_loss_total / len(val_loader)
            val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["val_f1"].append(val_metrics["macro_f1"])
            history["val_acc"].append(val_metrics["accuracy"])

            is_best = ""
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                torch.save(best_weights, ckpt_path)
                is_best = " ⭐ [Best Saved]"

            print(
                f"N={n_bins} Epoch {epoch:02d}/{epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
            )

        train_seconds = time.time() - start_time
        print(f"\n🎉 N={n_bins} Training Finished in {train_seconds:.1f}s | Best Val Macro-F1: {best_val_f1:.4f} "
              f"(epoch {best_epoch})")

        # --- 5. Unseen test evaluation with the best-val weights (as v5 Cell 7) ---
        if best_weights is not None:
            model.load_state_dict(best_weights)
        model.eval()
        test_preds, test_targets, test_probs = [], [], []
        with torch.no_grad():
            for batch in test_loader:
                xa = batch["x_a"].to(device)
                xs = batch["x_s"].to(device)
                ta = batch["tau_a"].to(device)
                ts = batch["tau_s"].to(device)
                labels = batch["label"].to(device)

                logits, _, _, _ = model(xa, xs, ta, ts)
                probs = F.softmax(logits, dim=-1)
                test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                test_targets.extend(labels.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())

        test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs))
        print("=" * 65)
        print(f"     BI-MODAL AEROSYNC-MAMBA (N={n_bins}): 100% REAL TEST BENCHMARK")
        print("=" * 65)
        print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
        print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
        print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
        print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
        print("=" * 65)
    finally:
        # --- 6. Free GPU memory so sequential conditions never accumulate ---
        del model, optimizer, scheduler, criterion, best_weights
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[GPU] freed | allocated={torch.cuda.memory_allocated() / 1024**2:.1f} MB "
                  f"| reserved={torch.cuda.memory_reserved() / 1024**2:.1f} MB")

    return {
        "N": n_bins,
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "train_seconds": round(train_seconds, 1),
        "best_val_macro_f1": round(float(best_val_f1), 6),
        "best_epoch": best_epoch,
        "freq_min_hz": round(prep["freq_min_hz"], 1) if prep["freq_min_hz"] is not None else None,
        "freq_max_hz": round(prep["freq_max_hz"], 1) if prep["freq_max_hz"] is not None else None,
        "seed": seed,
        "run_seconds": round(time.time() - t_run0, 1),
        # N=64 reproduction fingerprints (not written to the CSV)
        "_bin_indices": prep["bin_indices"].tolist(),
        "_n_windows": prep["n_windows"],
        "_class_counts": class_counts.tolist(),
        "_epoch1": {"train_loss": history["train_loss"][0], "val_loss": history["val_loss"][0],
                    "val_acc": history["val_acc"][0], "val_f1": history["val_f1"][0]},
    }

print(f"run_experiment(N) ready — sweep budgets: {N_BUDGETS}")

run_experiment(N) ready — sweep budgets: [32, 48, 64, 96, 128]


# Cell 7: Sweep Driver — N = 32, 48, 64, 96, 128 (ascending), CSV row after each, N=64 checked against v5

In [11]:
# ==============================================================================
# DRIVER — all 5 budgets in ascending order, one notebook execution. Each row is
# appended to the CSV the moment its condition finishes (timeout-safe). The N=64
# reproduction check against v5 prints IMMEDIATELY after N=64 finishes.
# ==============================================================================
import csv
import traceback
from datetime import datetime

SWEEP_CSV = Path("/kaggle/working/frequency_budget_sweep_results.csv")
SWEEP_CSV_FIELDS = ["N", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "train_seconds",
                    "best_val_macro_f1", "best_epoch", "freq_min_hz", "freq_max_hz", "seed",
                    "run_seconds", "note", "finished_at"]

# Known v5 result for this exact configuration at N=64 (Ran_minor-project-v5.ipynb).
V5_BASELINE_N64 = {"accuracy": 0.9009, "macro_f1": 0.8966, "balanced_accuracy": 0.8940, "macro_auroc": 0.9884}
# Deterministic v5 fingerprints — these do NOT depend on GPU numerics and must match EXACTLY.
V5_BIN_INDICES_N64 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                      258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                      509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                      761, 762, 763, 764, 765, 766]
V5_N_WINDOWS = (4934, 954, 1130)
V5_CLASS_COUNTS = [906, 1020, 1004, 1008, 996]
# v5 epoch-1 log line (first place GPU nondeterminism could show up).
V5_EPOCH1 = {"train_loss": 1.3863, "val_loss": 0.8762, "val_acc": 0.8249, "val_f1": 0.8390}
REPRO_TOL = 0.0005   # v5 numbers are reported to 4 decimals -> "near-identical" = within rounding

def check_n64_against_v5(res):
    """Loud reproduction check for N=64. Returns a one-line verdict for the CSV."""
    print("\n" + "=" * 90)
    print("  N=64 SANITY CHECK — does this run reproduce the verified v5 baseline?")
    print("=" * 90)

    det = {
        "strongest-64 bin indices": res["_bin_indices"] == V5_BIN_INDICES_N64,
        f"train/val/test windows {V5_N_WINDOWS}": tuple(res["_n_windows"]) == V5_N_WINDOWS,
        f"train class counts {V5_CLASS_COUNTS}": res["_class_counts"] == V5_CLASS_COUNTS,
    }
    for name, ok in det.items():
        print(f"  [deterministic] {name:<48}: {'MATCH ✅' if ok else 'MISMATCH ❌'}")
    if not det["strongest-64 bin indices"]:
        print(f"      this run: {res['_bin_indices']}")

    e1 = res["_epoch1"]
    print("  [epoch 1]       " + " | ".join(
        f"{k}: v5 {V5_EPOCH1[k]:.4f} vs now {e1[k]:.4f} ({e1[k] - V5_EPOCH1[k]:+.4f})" for k in V5_EPOCH1))

    print(f"\n  {'metric':<20} {'v5 known':>10} {'this run':>10} {'delta':>10}")
    worst = 0.0
    for k, v in V5_BASELINE_N64.items():
        d = res[k] - v
        worst = max(worst, abs(d))
        print(f"  {k:<20} {v:>10.4f} {res[k]:>10.4f} {d:>+10.4f}{'' if abs(d) <= REPRO_TOL else '   <-- differs'}")

    if not all(det.values()):
        verdict = "PORT BUG: deterministic v5 fingerprint mismatch"
        print("\n" + "🚨" * 40)
        print("🚨  N=64 DOES NOT REPRODUCE v5 — the bins / windows / class counts differ, which cannot be GPU")
        print("🚨  noise: something in the data path changed during the v5 -> v6 port. Do NOT trust this sweep")
        print("🚨  until that is fixed. (The remaining conditions still run so the session is not wasted.)")
        print("🚨" * 40)
    elif worst <= REPRO_TOL:
        verdict = "reproduces v5 (all 4 metrics within 0.0005)"
        print(f"\n✅ N=64 REPRODUCES v5: all four test metrics within ±{REPRO_TOL} (the 4-decimal rounding of the "
              f"reported v5 numbers) and all deterministic fingerprints match.")
    else:
        verdict = f"DOES NOT reproduce v5 metrics (max |delta| {worst:.4f}); data fingerprints match"
        print("\n" + "🚨" * 40)
        print(f"🚨  N=64 TEST METRICS DIFFER FROM v5 (largest |delta| = {worst:.4f} > {REPRO_TOL}).")
        print("🚨  Data path is identical (bins, windows and class counts all match), so the difference arose in")
        print("🚨  training. Check the epoch-1 line above: if epoch 1 already differs, the model/RNG path changed;")
        print("🚨  if epoch 1 matches and later epochs drift, it is GPU non-determinism (mamba_ssm CUDA kernels")
        print("🚨  are not bit-deterministic), which also limits how much any N-to-N gap below that size means.")
        print("🚨" * 40)
    print("=" * 90)
    return verdict

def append_sweep_row(res, note=""):
    row = {k: res.get(k) for k in SWEEP_CSV_FIELDS}
    row["note"] = note
    row["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    write_header = not SWEEP_CSV.exists()
    with open(SWEEP_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=SWEEP_CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(row)
    print(f"💾 appended N={res['N']} to {SWEEP_CSV} ({'header + row' if write_header else 'row'})")

if SWEEP_CSV.exists():
    _backup = SWEEP_CSV.with_suffix(f".{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak.csv")
    SWEEP_CSV.rename(_backup)
    print(f"ℹ️  existing results file moved to {_backup}")

sweep_failed = []
_sweep_t0 = time.time()
for i, n_bins in enumerate(sorted(N_BUDGETS), start=1):
    print("\n\n" + "=" * 90)
    print(f"=== Running N={n_bins} ({i}/{len(N_BUDGETS)}) ===   elapsed so far: {(time.time() - _sweep_t0) / 60:.1f} min")
    print("=" * 90)
    try:
        res = run_experiment(n_bins)
    except Exception as exc:
        sweep_failed.append((n_bins, repr(exc)))
        print(f"\n❌ N={n_bins} FAILED: {exc!r}")
        traceback.print_exc()
        if n_bins == 64:
            print("🚨" * 40 + "\n🚨  the N=64 v5-reproduction check could not run because N=64 failed.\n" + "🚨" * 40)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue
    note = check_n64_against_v5(res) if n_bins == 64 else ""
    append_sweep_row(res, note)

print("\n\n" + "=" * 90)
print(f"SWEEP COMPLETE — {len(N_BUDGETS) - len(sweep_failed)}/{len(N_BUDGETS)} conditions finished in "
      f"{(time.time() - _sweep_t0) / 60:.1f} min | results: {SWEEP_CSV}")
for n_bins, err in sweep_failed:
    print(f"  ❌ N={n_bins}: {err}")
print("=" * 90)



=== Running N=32 (1/5) ===   elapsed so far: 0.0 min
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[STRONGEST-32] Computing bin-selection statistics from TRAINING-SET audio only (69 training missions) — BEFORE any train/validation/test window preprocessing.


Bin selection N=32 (train-only audio scan):   0%|          | 0/69 [00:00<?, ?it/s]

[STRONGEST-32] Selected 32 bins from 356506 training-set audio chunks across 69 missions (FFT length = 769).
[STRONGEST-32] Selected bin indices (ascending): [2, 4, 5, 6, 7, 249, 250, 252, 254, 258, 260, 261, 262, 263, 264, 504, 505, 506, 507, 508, 510, 514, 516, 517, 518, 519, 520, 761, 762, 763, 764, 766]
[STRONGEST-32] Estimated audio sample rate ≈ 84695.4 Hz (derived from audio buffer length and real inter-buffer timing across training missions; the dataset does not document a mic sample rate directly, so this is a data-derived estimate for labeling only, not a hardcoded spec).
[STRONGEST-32] Selected bins as 13 contiguous band(s) -> approx. frequency (Hz):
    bins   2-  2 (  1 bins)  ->     110.3 -    110.3 Hz
    bins   4-  7 (  4 bins)  ->     220.6 -    386.0 Hz
    bins 249-250 (  2 bins)  ->   13729.9 -  13785.1 Hz
    bins 252-252 (  1 bins)  ->   13895.3 -  13895.3 Hz
    bins 254-254 (  1 bins)  ->   14005.6 -  14005.6 Hz
    bins 258-258 (  1 bins)  ->   14226.2 -  14226

Train Class 0 (N=32):   0%|          | 0/13 [00:00<?, ?it/s]

[STRONGEST-32 VERIFICATION] audio chunk shape (raw waveform): (5103, 1536) | np.fft.rfft output shape: (5103, 769) | final a_feats (sqrt-compressed, frozen strongest-32 bins) shape: (5103, 32)
[STRONGEST-32 VERIFICATION] Confirmed: features were produced via np.fft.rfft (frequency-domain magnitude spectrum) using the frozen strongest-32 bin indices, NOT the first 32 bins.


...

N=128 Epoch 18/20 | Train Loss: 0.2494 | Val Loss: 0.7274 | Val Acc: 0.9015 | Val Macro-F1: 0.9047


N=128 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

N=128 Epoch 19/20 | Train Loss: 0.2206 | Val Loss: 0.7272 | Val Acc: 0.9046 | Val Macro-F1: 0.9084


N=128 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

N=128 Epoch 20/20 | Train Loss: 0.2378 | Val Loss: 0.7648 | Val Acc: 0.8994 | Val Macro-F1: 0.9038

🎉 N=128 Training Finished in 233.5s | Best Val Macro-F1: 0.9114 (epoch 8)
     BI-MODAL AEROSYNC-MAMBA (N=128): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.8841 (88.41%)
  • Macro-F1 Score      : 0.8800
  • Balanced Accuracy   : 0.8740
  • One-vs-Rest AUROC   : 0.9860
[GPU] freed | allocated=37.4 MB | reserved=88.0 MB
💾 appended N=128 to /kaggle/working/frequency_budget_sweep_results.csv (row)


SWEEP COMPLETE — 5/5 conditions finished in 33.3 min | results: /kaggle/working/frequency_budget_sweep_results.csv


# Cell 8: Final Comparison Table — sorted by macro-F1, N=64 vs v5, winning N

In [12]:
# ==============================================================================
# FINAL COMPARISON TABLE — reloaded from the CSV on disk (crash-safe source of
# truth), sorted by test macro-F1 descending, N=64 compared against v5, winner
# identified. Works on partial results too.
# ==============================================================================
if not SWEEP_CSV.exists():
    raise FileNotFoundError(f"{SWEEP_CSV} does not exist — no condition finished. Re-run the driver cell.")

sweep_df = pd.read_csv(SWEEP_CSV).sort_values("macro_f1", ascending=False, kind="stable").reset_index(drop=True)
winner_N = int(sweep_df.iloc[0]["N"])
sweep_df.insert(0, "rank", np.arange(1, len(sweep_df) + 1))
sweep_df["marker"] = [("🏆 WINNER" if int(n) == winner_N else "") + (" (v5 baseline N)" if int(n) == 64 else "")
                      for n in sweep_df["N"]]

table = sweep_df[["rank", "N", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "train_seconds",
                  "freq_min_hz", "freq_max_hz", "best_val_macro_f1", "best_epoch", "marker"]].copy()
for col in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"):
    table[col] = table[col].round(4)

pd.set_option("display.width", 250)
print("=" * 110)
print(f"   FREQUENCY-BUDGET SWEEP — strongest-N, N ∈ {N_BUDGETS} — sorted by test macro-F1 (descending)")
print(f"   ({len(sweep_df)}/{len(N_BUDGETS)} conditions in the CSV)")
print("=" * 110)
try:
    from IPython.display import display
    display(table)
except Exception:
    print(table.to_string(index=False))
print("=" * 110)

n64 = sweep_df[sweep_df["N"] == 64]
if len(n64):
    r = n64.iloc[0]
    print("N=64 vs known v5 baseline:")
    for k, v in V5_BASELINE_N64.items():
        print(f"  {k:<20} v5 {v:.4f} | this run {r[k]:.4f} | delta {r[k] - v:+.4f}")
    print(f"  reproduction verdict: {r['note']}")
else:
    print("🚨 N=64 is not in the CSV — the v5 reproduction check did not complete.")

w = sweep_df.iloc[0]
d64 = f" | Δ macro-F1 vs this run's N=64: {w['macro_f1'] - n64.iloc[0]['macro_f1']:+.4f}" if len(n64) else ""
print(f"\n🏆 WINNING N = {winner_N}: test macro-F1 {w['macro_f1']:.4f} | accuracy {w['accuracy']:.4f} | "
      f"balanced acc {w['balanced_accuracy']:.4f} | macro-AUROC {w['macro_auroc']:.4f}{d64}")
if len(sweep_df) < len(N_BUDGETS):
    print(f"⚠️  only {len(sweep_df)}/{len(N_BUDGETS)} conditions finished — the winner is among those only.")
print(f"📄 full results CSV: {SWEEP_CSV}")

   FREQUENCY-BUDGET SWEEP — strongest-N, N ∈ [32, 48, 64, 96, 128] — sorted by test macro-F1 (descending)
   (5/5 conditions in the CSV)


,rank,N,accuracy,macro_f1,balanced_accuracy,macro_auroc,train_seconds,freq_min_hz,freq_max_hz,best_val_macro_f1,best_epoch,marker
0,1,64,0.9009,0.8966,0.8940,0.9884,212.9,55.1,42237.4,0.908911,17,🏆 WINNER (v5 baseline N)
1,2,48,0.8903,0.8863,0.8837,0.9860,301.3,110.3,42237.4,0.891042,16,
2,3,96,0.8876,0.8824,0.8791,0.9866,272.4,0.0,42347.7,0.910372,14,
3,4,128,0.8841,0.8800,0.8740,0.9860,233.5,0.0,42347.7,0.911449,8,
4,5,32,0.8770,0.8717,0.8714,0.9859,210.8,110.3,42237.4,0.897469,9,


N=64 vs known v5 baseline:
  accuracy             v5 0.9009 | this run 0.9009 | delta -0.0000
  macro_f1             v5 0.8966 | this run 0.8966 | delta +0.0000
  balanced_accuracy    v5 0.8940 | this run 0.8940 | delta -0.0000
  macro_auroc          v5 0.9884 | this run 0.9884 | delta -0.0000
  reproduction verdict: reproduces v5 (all 4 metrics within 0.0005)

🏆 WINNING N = 64: test macro-F1 0.8966 | accuracy 0.9009 | balanced acc 0.8940 | macro-AUROC 0.9884 | Δ macro-F1 vs this run's N=64: +0.0000
📄 full results CSV: /kaggle/working/frequency_budget_sweep_results.csv
